# Week 5 Spark DataFrame Assignment

## Objective

Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using Spark DataFrames.

This notebook contains both theory answers and PySpark code solutions for all 15 assignment questions.

In [1]:
# Import SparkSession to create Spark application
from pyspark.sql import SparkSession

# Import commonly used PySpark SQL functions
from pyspark.sql.functions import (
    col,
    avg,
    count,
    sum,
    min,
    max,
    mean,
    when
)

# Import data types for schema modification
from pyspark.sql.types import TimestampType

In [2]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Week 5 Spark DataFrame Assignment") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


In [ ]:
# Load the CSV dataset
df = spark.read.csv(
    "data/sample_sales_data.csv",
    header=True,
    inferSchema=True
)
# Display dataset
df.show()
# Display schema
df.printSchema()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|      raw_timestamp|          email| username|price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|    101|      2026-01-01|  West|     Electronics|       1200|   Mumbai| 25|     Premium|  Active|2026-01-01 10:30:00|user1@gmail.com|  userone| 1200|      S1|       2|
|    102|      2026-01-02|  East|       Furniture|        800|  Kolkata| 31|       Basic|    NULL|2026-01-02 11:00:00|user2@gmail.com|  usertwo|  800|      S2|       1|
|    103|      2026-01-03|  West|     Electronics|       1500|    Delhi| 22|     Premium|  Active|2026-01-03 12:15:00|           NULL|userthree| 1500|     

## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce has several limitations:

1. MapReduce writes intermediate results to disk after every operation, which makes it slower.
2. It is not efficient for iterative tasks like machine learning because data has to be read and written repeatedly.
3. The programming model is more complex because developers need to write separate map and reduce logic.
4. It is mostly suitable for batch processing and not ideal for real-time or near real-time processing.
5. It has higher latency compared to Spark.

Spark is preferred because it supports in-memory processing, faster execution, easier APIs, DataFrames, SQL, streaming, and machine learning libraries.

## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark uses in-memory computing by storing intermediate data in RAM instead of writing it to disk after every step.

Machine learning algorithms usually run multiple iterations on the same dataset. In disk-based systems like MapReduce, each iteration reads data from disk and writes results back to disk. This makes the process slow.

Spark can cache or persist the data in memory, so repeated operations become faster.

Example:

```python
df.cache()

## Q3: Remove duplicate rows based on user_id and transaction_date

In this step, duplicate records are removed using the `dropDuplicates()` function.  
The duplicate check is performed only on the `user_id` and `transaction_date` columns.

In [4]:
df_no_duplicates = df.dropDuplicates(["user_id", "transaction_date"])

df_no_duplicates.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|      raw_timestamp|          email| username|price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|    101|      2026-01-01|  West|     Electronics|       1200|   Mumbai| 25|     Premium|  Active|2026-01-01 10:30:00|user1@gmail.com|  userone| 1200|      S1|       2|
|    102|      2026-01-02|  East|       Furniture|        800|  Kolkata| 31|       Basic|    NULL|2026-01-02 11:00:00|user2@gmail.com|  usertwo|  800|      S2|       1|
|    103|      2026-01-03|  West|     Electronics|       1500|    Delhi| 22|     Premium|  Active|2026-01-03 12:15:00|           NULL|userthree| 1500|     

## Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In this question, the dataset is first filtered for the `West` region.

After filtering, the data is grouped by `product_category`, and the average sale amount is calculated using `avg()`.

In [5]:
west_avg_sales = df.filter(col("region") == "West") \
    .groupBy("product_category") \
    .agg(avg("sale_amount").alias("average_sale_amount"))

west_avg_sales.show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|     Electronics|             1300.0|
|        Clothing|              700.0|
|       Furniture|              925.0|
+----------------+-------------------+



## Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

`.na.drop()` is used to remove rows that contain null values.

`.na.fill()` is used to replace null values with a specific value.

For example, if the `status` column has missing values, we can replace them with `Unknown` using `.na.fill()`.

In [6]:
df_status_filled = df.na.fill({"status": "Unknown"})

df_status_filled.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|      raw_timestamp|          email| username|price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|    101|      2026-01-01|  West|     Electronics|       1200|   Mumbai| 25|     Premium|  Active|2026-01-01 10:30:00|user1@gmail.com|  userone| 1200|      S1|       2|
|    102|      2026-01-02|  East|       Furniture|        800|  Kolkata| 31|       Basic| Unknown|2026-01-02 11:00:00|user2@gmail.com|  usertwo|  800|      S2|       1|
|    103|      2026-01-03|  West|     Electronics|       1500|    Delhi| 22|     Premium|  Active|2026-01-03 12:15:00|           NULL|userthree| 1500|     

## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In this question, records are grouped by city using `groupBy()`.

Then the total number of records for each city is calculated using `count()`.

Finally, only cities with count greater than 100 are selected using `filter()`.

In [7]:
city_count = df.groupBy("city") \
    .agg(count("*").alias("total_records")) \
    .filter(col("total_records") > 100)

city_count.show()

+----+-------------+
|city|total_records|
+----+-------------+
+----+-------------+



### Note

Since the sample dataset is small, the output may be empty. This is acceptable because the logic is correct.

## Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

Spark DataFrames are immutable. This means the original DataFrame cannot be changed directly.

Whenever we perform operations like dropping columns, renaming columns, filtering rows, or filling null values, Spark creates a new DataFrame.

Example:

```python
df_new = df.drop("old_column")

## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 inclusive and the subscription is 'Premium'.

In this question, we apply multiple filter conditions:

1. Age should be greater than or equal to 18.
2. Age should be less than or equal to 30.
3. Subscription should be `Premium`.

In [8]:
filtered_df = df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
)

filtered_df.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|      raw_timestamp|          email| username|price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|    101|      2026-01-01|  West|     Electronics|       1200|   Mumbai| 25|     Premium|  Active|2026-01-01 10:30:00|user1@gmail.com|  userone| 1200|      S1|       2|
|    103|      2026-01-03|  West|     Electronics|       1500|    Delhi| 22|     Premium|  Active|2026-01-03 12:15:00|           NULL|userthree| 1500|      S1|       1|
|    101|      2026-01-01|  West|     Electronics|       1200|   Mumbai| 25|     Premium|  Active|2026-01-01 10:30:00|user1@gmail.com|  userone| 1200|     

## Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

It is better to handle null values before performing mathematical aggregations because null values can affect the accuracy of results.

For example, if the `price` column contains null values, then functions like `sum()` or `avg()` may ignore those values. This can produce incomplete or misleading results.

By filling or removing null values before aggregation, we get cleaner and more reliable results.

Example:

```python
df_cleaned = df.na.fill({"price": 0})

## Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In this question, the `raw_timestamp` column is converted to `TimestampType`.

After casting, the column is renamed from `raw_timestamp` to `event_time`.

In [9]:
df_timestamp = df.withColumn(
    "raw_timestamp",
    col("raw_timestamp").cast(TimestampType())
).withColumnRenamed("raw_timestamp", "event_time")

df_timestamp.printSchema()
df_timestamp.show()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|         event_time|          email| username|price|store_id|quantity|
+-------+----------------+------+------------

## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

Shuffle is the process where Spark moves data across partitions so that similar keys can be grouped together.

For example, during a `groupBy("city")`, Spark needs to bring all records of the same city together before calculating count, sum, or average.

This is considered a wide transformation because data from multiple partitions is exchanged and reorganized across the cluster.

Shuffle is expensive because it involves network transfer, disk usage, and extra processing time.

Example:

```python
df.groupBy("city").count()

## Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In this question, invalid user records are removed.

The DataFrame keeps only those rows where:

1. The `email` column is not null.
2. The `username` column is not an empty string.

In [10]:
df_cleaned_users = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

df_cleaned_users.show()

+-------+----------------+------+----------------+-----------+-------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   city|age|subscription|  status|      raw_timestamp|          email| username|price|store_id|quantity|
+-------+----------------+------+----------------+-----------+-------+---+------------+--------+-------------------+---------------+---------+-----+--------+--------+
|    101|      2026-01-01|  West|     Electronics|       1200| Mumbai| 25|     Premium|  Active|2026-01-01 10:30:00|user1@gmail.com|  userone| 1200|      S1|       2|
|    102|      2026-01-02|  East|       Furniture|        800|Kolkata| 31|       Basic|    NULL|2026-01-02 11:00:00|user2@gmail.com|  usertwo|  800|      S2|       1|
|    101|      2026-01-01|  West|     Electronics|       1200| Mumbai| 25|     Premium|  Active|2026-01-01 10:30:00|user1@gmail.com|  userone| 1200|      S1|       2

## Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

The `.agg()` function is used to calculate multiple aggregation values in a single query.

Here, we calculate:

1. Minimum price
2. Maximum price
3. Average price

In [11]:
price_stats = df.agg(
    min("price").alias("minimum_price"),
    max("price").alias("maximum_price"),
    mean("price").alias("average_price")
)

price_stats.show()

+-------------+-------------+------------------+
|minimum_price|maximum_price|     average_price|
+-------------+-------------+------------------+
|          500|         2000|1105.5555555555557|
+-------------+-------------+------------------+



## Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

When `inferSchema=True` is used, Spark automatically guesses the data type of each column.

If a date column contains messy or inconsistent formats, Spark may infer the wrong data type. For example, it may treat the date column as a string instead of a date or timestamp.

This can create problems during filtering, sorting, or date-based calculations.

A better approach is to define the schema manually when the dataset contains inconsistent formats.

Example:

```python
df = spark.read.csv("data.csv", header=True, inferSchema=True)

## Q15: Write a final processing pipeline that:

- Filters out duplicates
- Fills null prices with 0
- Groups by store_id to calculate total revenue

In this final pipeline, multiple data processing steps are combined together.

The pipeline performs:

1. Duplicate removal
2. Null price replacement with 0
3. Revenue calculation using `price * quantity`
4. Grouping by `store_id`
5. Total revenue calculation

In [12]:
final_pipeline = df.dropDuplicates() \
    .na.fill({"price": 0}) \
    .withColumn("revenue", col("price") * col("quantity")) \
    .groupBy("store_id") \
    .agg(sum("revenue").alias("total_revenue"))

final_pipeline.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|      S3|         2900|
|      S2|         1750|
|      S1|         7000|
+--------+-------------+



### Final Insight

The final pipeline combines cleaning, transformation, and aggregation in one flow.

It removes duplicate rows, handles missing prices, calculates revenue, and gives total revenue for each store.

In [13]:
spark.stop()